In [1]:
# ============================================================
# Experimental Design and Implementation Choices
# ============================================================

# Reproducibility and Randomness Control
# - fixed seed reproducibility
# - deterministic pseudo-random number generation using NumPy RNG
# - independent RNG streams for each agent
# - controlled randomness for environment generation and reward sampling

# Bandit Environment Setup
# - new bandit instance generated at the start of each run
# - true action values q*(a) sampled from a Gaussian distribution
# - reward samples drawn from a Gaussian distribution with mean q*(a) and variance 1
# - SAME q*(a) shared across all agents within a run
# - independent environment copies created for each agent
# - INDEPENDENT reward samples per agent within a run
# - random tie-breaking for greedy action selection

# Problem 1: Greedy vs ε-Greedy Evaluation
# - stationary k-armed bandit testbed
# - greedy policy (ε = 0)
# - ε-greedy exploration policies (ε = 0.01 and ε = 0.1)
# - sample-average action-value estimation
# - comparison of exploration vs exploitation behaviour
# - sensitivity analysis across different arm counts (n = 5, 10, 20)

# Problem 2: Exploration Strategy Comparison
# - ε-greedy exploration strategy
# - optimistic initial values strategy
# - upper confidence bound (UCB) action selection
# - softmax (Boltzmann) exploration
# - gradient bandit algorithm with reward baseline
# - parameter sensitivity analysis for exploration parameters
# - logarithmic parameter scaling used for parameter study plots

# Problem 3: Action-Value Estimation Methods
# - sample-average estimator
# - constant step-size estimator
# - unbiased constant step-size estimator
# - sliding window estimator
# - ε-greedy action selection used for all estimators

# Stationary Bandit Experiments
# - stationary Gaussian multi-armed bandit testbed
# - learning curves generated for average reward and % optimal action
# - expected performance estimated through averaging across runs

# Non-Stationary Bandit Experiments
# - Gaussian random walk used to model drifting action values
# - SHARED non-stationary drift across estimators
# - optimal action recomputed at each step
# - evaluation of estimator adaptability in changing environments

# Experiment Evaluation Methodology
# - independent bandit runs used to approximate expected reward E[R] of each algorithm
# - results averaged across multiple independent runs
# - average reward curves computed across runs
# - percentage optimal action curves computed across runs
# - normalized AUC computed as mean reward over time
# - final performance measured using last-step reward

# Sensitivity and Scaling Analysis
# - multi-armed bandit sizes evaluated for n = 5, 10, 20
# - parameter sensitivity analysis across exploration strategies

# Output Generation
# - learning curve comparison figures generated
# - parameter sensitivity plots generated
# - summary tables exported as CSV files
# - experiment outputs saved for reproducibility
# ============================================================

In [2]:
# ============================================================
# Libraries
# ============================================================

import os
import csv
from dataclasses import dataclass
from typing import Dict, Callable, Tuple, List, Optional

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
# ------------------------------------------------------------
# Smoothing helper for plotting learning curves
#
# Bandit rewards are stochastic (R_t ~ N(q*(a), 1)), which can
# produce small fluctuations in the averaged learning curves
# even after averaging across many runs. To improve readability
# of the plots, we apply a simple moving-average smoothing
# window when visualizing the results.
#
# Important:
# - Smoothing is applied ONLY for visualization.
# - The underlying experimental results, statistics, and tables
#   are computed using the raw data.
# - This helps reveal the overall learning trends of different
#   algorithms without altering the experiment itself.
# ------------------------------------------------------------

def smooth(x, window=50):
    """
    Moving-average smoothing for visualization of learning curves.
    Uses edge padding to avoid artificial drops at the boundaries.
    """
    pad = window // 2
    x_padded = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(x_padded, kernel, mode="valid")

In [4]:
# ============================================================
# Shared Helpers
# ============================================================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def argmax_random_tie(values: np.ndarray, rng: np.random.Generator) -> int:
    max_val = np.max(values)
    candidates = np.flatnonzero(values == max_val)
    return int(rng.choice(candidates))


def softmax(x: np.ndarray) -> np.ndarray:
    z = x - np.max(x)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)


def final_and_auc(curve: np.ndarray) -> Tuple[float, float]:
    """
    Final value + normalized AUC (mean over time).
    """
    return float(curve[-1]), float(np.mean(curve))


def print_table(rows: List[dict], title: str) -> None:
    if not rows:
        print(f"\n{title}\n{'-' * len(title)}")
        print("(No rows)")
        return

    print("\n" + title)
    print("-" * len(title))

    cols = list(rows[0].keys())
    widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) for c in cols}

    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)

    print(header)
    print(sep)
    for r in rows:
        print(" | ".join(str(r[c]).ljust(widths[c]) for c in cols))


In [5]:
# ============================================================
# Shared Bandit Environments
# ============================================================

@dataclass
class StationaryBandit:
    n_arms: int
    q_star_mean: float = 0.0
    q_star_std: float = 1.0
    reward_std: float = 1.0

    def reset(self, rng: np.random.Generator) -> np.ndarray:
        self.q_star = rng.normal(self.q_star_mean, self.q_star_std, size=self.n_arms)
        return self.q_star

    def set_q_star(self, q_star: np.ndarray) -> None:
        self.q_star = np.array(q_star, dtype=float).copy()

    def step(self, action: int, rng: np.random.Generator) -> float:
        return float(rng.normal(loc=self.q_star[action], scale=self.reward_std))

    def optimal_action(self) -> int:
        return int(np.argmax(self.q_star))


@dataclass
class NonStationaryBandit(StationaryBandit):
    random_walk_std: float = 0.01

    def step(self, action: int, rng: np.random.Generator) -> float:
        return float(rng.normal(loc=self.q_star[action], scale=self.reward_std))

    def apply_shared_drift(self, drift: np.ndarray) -> None:
        self.q_star = self.q_star + drift


def make_agent_envs(
    env_class,
    n_arms: int,
    q_star: np.ndarray,
    agent_names: List[str],
    **env_kwargs
) -> Dict[str, object]:
    """
    Create one environment per agent, all initialized with the same q_star.
    """
    envs = {}
    for name in agent_names:
        env = env_class(n_arms=n_arms, **env_kwargs)
        env.set_q_star(q_star)
        envs[name] = env
    return envs


In [6]:
# ============================================================
# Problem 1
# Greedy / epsilon-greedy with sample-average estimator
# ============================================================

class EpsilonGreedySampleAverageAgent:
    def __init__(self, n_arms: int, epsilon: float, initial: float = 0.0):
        self.n_arms = n_arms
        self.epsilon = float(epsilon)
        self.initial = float(initial)
        self.reset()

    def reset(self) -> None:
        self.Q = np.full(self.n_arms, self.initial, dtype=float)
        self.N = np.zeros(self.n_arms, dtype=int)

    def select_action(self, rng: np.random.Generator) -> int:
        if rng.random() < self.epsilon:
            return int(rng.integers(0, self.n_arms))
        return argmax_random_tie(self.Q, rng)

    def update(self, action: int, reward: float) -> None:
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


def run_problem1_experiment(
    n_arms: int,
    n_steps: int,
    n_runs: int,
    methods: Dict[str, Callable[[], EpsilonGreedySampleAverageAgent]],
    seed: int = 42
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    master_rng = np.random.default_rng(seed)

    avg_reward = {name: np.zeros(n_steps, dtype=float) for name in methods}
    pct_opt = {name: np.zeros(n_steps, dtype=float) for name in methods}

    method_names = list(methods.keys())

    for run in range(n_runs):
        q_star = master_rng.normal(0.0, 1.0, size=n_arms)
        optimal = int(np.argmax(q_star))

        envs = make_agent_envs(
            env_class=StationaryBandit,
            n_arms=n_arms,
            q_star=q_star,
            agent_names=method_names
        )

        agents = {name: ctor() for name, ctor in methods.items()}
        agent_rngs = {
            name: np.random.default_rng(seed + 100000 * (run + 1) + i)
            for i, name in enumerate(method_names)
        }

        for t in range(n_steps):
            for name, agent in agents.items():
                rng = agent_rngs[name]
                action = agent.select_action(rng)
                reward = envs[name].step(action, rng)
                agent.update(action, reward)

                avg_reward[name][t] += reward
                pct_opt[name][t] += 1.0 if action == optimal else 0.0

    for name in methods:
        avg_reward[name] /= n_runs
        pct_opt[name] = 100.0 * pct_opt[name] / n_runs

    return avg_reward, pct_opt


def plot_problem1_curves(
    avg_reward: Dict[str, np.ndarray],
    pct_opt: Dict[str, np.ndarray],
    out_dir: str = "outputs"
) -> None:
    ensure_dir(out_dir)

    plt.figure(figsize=(9, 5))
    for name, curve in avg_reward.items():
        plt.plot(smooth(curve), label=name,linewidth=2.5)
    plt.title("Problem 1: Average Reward vs Steps (Stationary 10-armed bandit)")
    plt.xlabel("Steps")
    plt.ylabel("Average reward")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "fig1_q1_eps_greedy_avg_reward.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(9, 5))
    for name, curve in pct_opt.items():
        plt.plot(smooth(curve), label=name,linewidth=2.5)
    plt.title("Problem 1: % Optimal Action vs Steps (Stationary 10-armed bandit)")
    plt.xlabel("Steps")
    plt.ylabel("% Optimal action")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "fig1_q1_eps_greedy_pct_optimal.png"), dpi=200)
    plt.close()


def build_problem1_summary_table(
    n_values: List[int],
    n_steps: int,
    n_runs: int,
    seed: int = 42,
    out_dir: str = "outputs"
) -> List[dict]:
    ensure_dir(out_dir)
    rows = []

    for n in n_values:
        methods = {
            "Greedy (ε=0)": lambda n=n: EpsilonGreedySampleAverageAgent(n_arms=n, epsilon=0.0),
            "ε-greedy (ε=0.01)": lambda n=n: EpsilonGreedySampleAverageAgent(n_arms=n, epsilon=0.01),
            "ε-greedy (ε=0.1)": lambda n=n: EpsilonGreedySampleAverageAgent(n_arms=n, epsilon=0.1),
        }

        avg_reward, pct_opt = run_problem1_experiment(
            n_arms=n,
            n_steps=n_steps,
            n_runs=n_runs,
            methods=methods,
            seed=seed
        )

        scores = []
        for name, reward_curve in avg_reward.items():
            final_reward, auc_reward = final_and_auc(reward_curve)
            final_opt, auc_opt = final_and_auc(pct_opt[name])

            scores.append({
                "name": name,
                "final_reward": final_reward,
                "auc_reward": auc_reward,
                "std_reward_curve": float(np.std(reward_curve)),
                "final_opt": final_opt,
                "auc_opt": auc_opt,
                "std_opt_curve": float(np.std(pct_opt[name]))
            })

        scores.sort(key=lambda x: x["auc_reward"], reverse=True)

        best = scores[0]
        runner_up = scores[1]

        rows.append({
            "n": n,
            "Best Policy": best["name"],
            "Runner-Up Policy": runner_up["name"],
            "Final Avg Reward": round(best["final_reward"], 4),
            "AUC Avg Reward": round(best["auc_reward"], 4),
            "Std Avg Reward": round(best["std_reward_curve"], 4),
            "Final % Optimal": round(best["final_opt"], 2),
            "AUC % Optimal": round(best["auc_opt"], 2),
            "Std % Optimal": round(best["std_opt_curve"], 2),
        })

    csv_path = os.path.join(out_dir, "table_problem1_n_sensitivity.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    return rows


def run_problem1(out_dir: str, n_steps: int, n_runs: int, seed: int) -> None:
    main_n = 10

    methods = {
        "Greedy (ε=0)": lambda: EpsilonGreedySampleAverageAgent(n_arms=main_n, epsilon=0.0),
        "ε-greedy (ε=0.01)": lambda: EpsilonGreedySampleAverageAgent(n_arms=main_n, epsilon=0.01),
        "ε-greedy (ε=0.1)": lambda: EpsilonGreedySampleAverageAgent(n_arms=main_n, epsilon=0.1),
    }

    avg_reward, pct_opt = run_problem1_experiment(
        n_arms=main_n,
        n_steps=n_steps,
        n_runs=n_runs,
        methods=methods,
        seed=seed
    )

    plot_problem1_curves(avg_reward, pct_opt, out_dir=out_dir)

    rows = build_problem1_summary_table(
        n_values=[5, 10, 20],
        n_steps=n_steps,
        n_runs=n_runs,
        seed=seed,
        out_dir=out_dir
    )

    print_table(rows, "Problem 1 Summary Table: Sensitivity to Number of Arms")

In [7]:
# ============================================================
# Problem 2
# Exploration vs Exploitation
# ============================================================

class QAgentBase:
    def __init__(self, n_arms: int, initial: float = 0.0):
        self.n_arms = n_arms
        self.initial = initial
        self.reset()

    def reset(self):
        self.Q = np.full(self.n_arms, self.initial, dtype=float)
        self.N = np.zeros(self.n_arms, dtype=int)

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        raise NotImplementedError

    def update(self, action: int, reward: float) -> None:
        raise NotImplementedError


class EpsilonGreedyAgent(QAgentBase):
    def __init__(self, n_arms: int, epsilon: float = 0.1, initial: float = 0.0):
        self.epsilon = epsilon
        super().__init__(n_arms, initial)

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        if rng.random() < self.epsilon:
            return int(rng.integers(0, self.n_arms))
        return argmax_random_tie(self.Q, rng)

    def update(self, action: int, reward: float) -> None:
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class OptimisticGreedyAgent(QAgentBase):
    def __init__(self, n_arms: int, q0: float = 5.0, alpha: float = 0.1):
        self.alpha = alpha
        super().__init__(n_arms, q0)

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        return argmax_random_tie(self.Q, rng)

    def update(self, action: int, reward: float) -> None:
        self.Q[action] += self.alpha * (reward - self.Q[action])


class UCBAgent(QAgentBase):
    def __init__(self, n_arms: int, c: float = 1.0, initial: float = 0.0):
        self.c = c
        super().__init__(n_arms, initial)

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(rng.choice(untried))
        bonus = self.c * np.sqrt(np.log(t + 1) / self.N)
        return argmax_random_tie(self.Q + bonus, rng)

    def update(self, action: int, reward: float) -> None:
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class SoftmaxAgent(QAgentBase):
    def __init__(self, n_arms: int, tau: float = 0.1, initial: float = 0.0):
        self.tau = tau
        super().__init__(n_arms, initial)

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        probs = softmax(self.Q / self.tau)
        return int(rng.choice(self.n_arms, p=probs))

    def update(self, action: int, reward: float) -> None:
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class GradientBanditAgent:
    def __init__(self, n_arms: int, alpha: float = 0.1, use_baseline: bool = True):
        self.n_arms = n_arms
        self.alpha = alpha
        self.use_baseline = use_baseline
        self.reset()

    def reset(self):
        self.H = np.zeros(self.n_arms)
        self.pi = np.ones(self.n_arms) / self.n_arms
        self.avg_reward = 0.0
        self.t = 0

    def select_action(self, t: int, rng: np.random.Generator) -> int:
        self.pi = softmax(self.H)
        return int(rng.choice(self.n_arms, p=self.pi))

    def update(self, action: int, reward: float):
        self.t += 1
        baseline = self.avg_reward if self.use_baseline else 0.0
        if self.use_baseline:
            self.avg_reward += (reward - self.avg_reward) / self.t
        advantage = reward - baseline
        one_hot = np.zeros(self.n_arms)
        one_hot[action] = 1.0
        self.H += self.alpha * advantage * (one_hot - self.pi)


def run_learning_curves(
    n_arms: int,
    n_steps: int,
    n_runs: int,
    agents: Dict[str, Callable[[], object]],
    seed: int = 42
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    avg_reward = {name: np.zeros(n_steps) for name in agents}
    pct_opt = {name: np.zeros(n_steps) for name in agents}

    master_rng = np.random.default_rng(seed)
    agent_names = list(agents.keys())

    for run in range(n_runs):
        q_star = master_rng.normal(0.0, 1.0, size=n_arms)
        optimal = int(np.argmax(q_star))

        envs = make_agent_envs(
            env_class=StationaryBandit,
            n_arms=n_arms,
            q_star=q_star,
            agent_names=agent_names
        )

        agent_instances = {name: ctor() for name, ctor in agents.items()}
        rngs = {
            name: np.random.default_rng(seed + 100000 * (run + 1) + i)
            for i, name in enumerate(agent_names)
        }

        for t in range(n_steps):
            for name, agent in agent_instances.items():
                rng = rngs[name]
                action = agent.select_action(t, rng)
                reward = envs[name].step(action, rng)
                agent.update(action, reward)
                avg_reward[name][t] += reward
                pct_opt[name][t] += 1 if action == optimal else 0

    for name in avg_reward:
        avg_reward[name] /= n_runs
        pct_opt[name] = 100 * pct_opt[name] / n_runs

    return avg_reward, pct_opt


def parameter_study(
    n_arms: int,
    n_steps: int,
    n_runs: int,
    agents_grid: Dict[str, List[Callable[[], object]]],
    seed: int = 42
) -> Dict[str, Tuple[List[float], List[float]]]:
    results = {}
    for name, ctor_list in agents_grid.items():
        avg_rewards = []
        param_values = []

        for ctor in ctor_list:
            avg_reward = run_learning_curves(
                n_arms=n_arms,
                n_steps=n_steps,
                n_runs=n_runs,
                agents={name: ctor},
                seed=seed
            )[0][name].mean()

            avg_rewards.append(avg_reward)

            instance = ctor()
            param = getattr(
                instance,
                "epsilon",
                getattr(
                    instance,
                    "c",
                    getattr(
                        instance,
                        "tau",
                        getattr(instance, "alpha", None)
                    )
                )
            )

            if isinstance(instance, OptimisticGreedyAgent):
                param = instance.initial

            param_values.append(param)

        results[name] = (param_values, avg_rewards)

    return results


def plot_two_panel(avg_reward, pct_opt, filename_prefix, out_dir="outputs"):
    ensure_dir(out_dir)

    plt.figure(figsize=(10, 4))
    for name, curve in avg_reward.items():
        plt.plot(smooth(curve), label=name,linewidth=2.5)
    plt.title("Algorithm Comparison — Average Reward")
    plt.xlabel("Steps")
    plt.ylabel("Average Reward")
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_avg_reward.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(10, 4))
    for name, curve in pct_opt.items():
        plt.plot(smooth(curve), label=name,linewidth=2.5)
    plt.title("Algorithm Comparison — % Optimal Action")
    plt.xlabel("Steps")
    plt.ylabel("% Optimal Action")
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_pct_optimal.png"), dpi=200)
    plt.close()


def plot_parameter_sensitivity(results, filename, out_dir="outputs"):
    ensure_dir(out_dir)

    plt.figure(figsize=(10, 4))
    for name, (params, avg_rewards) in results.items():
        plt.plot(params, avg_rewards, marker='o', label=name)
    plt.xscale('log', base=2)
    plt.title("Parameter Sensitivity — Average Reward")
    plt.xlabel("Parameter Value (log2 scale)")
    plt.ylabel("Average Reward")
    plt.grid(True, alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=200)
    plt.close()


def run_problem2(out_dir: str, n_steps: int, n_runs: int, seed: int) -> None:
    n_arms = 10

    agents = {
        "ε-greedy (ε=0.1)": lambda: EpsilonGreedyAgent(n_arms, epsilon=0.1),
        "Optimistic (Q0=5, α=0.1)": lambda: OptimisticGreedyAgent(n_arms, q0=5.0, alpha=0.1),
        "UCB (c=1)": lambda: UCBAgent(n_arms, c=1.0),
        "Softmax (τ=0.1)": lambda: SoftmaxAgent(n_arms, tau=0.1),
        "Gradient bandit (α=0.1)": lambda: GradientBanditAgent(n_arms, alpha=0.1)
    }

    avg_reward, pct_opt = run_learning_curves(n_arms, n_steps, n_runs, agents, seed=seed)
    plot_two_panel(avg_reward, pct_opt, "fig2_q2_algorithm_comparison", out_dir)

    agents_grid = {
        "ε-greedy": [lambda eps=eps: EpsilonGreedyAgent(n_arms, epsilon=eps)
                     for eps in [1/128, 1/64, 1/32, 1/16, 1/8, 1/4]],
        "UCB": [lambda c=c: UCBAgent(n_arms, c=c)
                for c in [1/16, 1/8, 1/4, 1/2, 1, 2, 4]],
        "Optimistic": [lambda q0=q0: OptimisticGreedyAgent(n_arms, q0=q0, alpha=0.1)
                       for q0 in [1/4, 1/2, 1, 2, 4, 8]],
        "Softmax": [lambda tau=tau: SoftmaxAgent(n_arms, tau=tau)
                    for tau in [1/64, 1/32, 1/16, 1/8, 1/4, 1/2]],
        "Gradient bandit": [lambda alpha=a: GradientBanditAgent(n_arms, alpha=a)
                            for a in [1/64, 1/32, 1/16, 1/8, 1/4, 1/2]]
    }

    results = parameter_study(n_arms, n_steps, n_runs, agents_grid, seed=seed)
    plot_parameter_sensitivity(results, "fig3_q2_parameter_sensitivity.png", out_dir)

    table_2a = []
    for name, (params, avg_rewards) in results.items():
        idx = int(np.argmax(avg_rewards))
        table_2a.append({
            "Agent": name,
            "Best Parameter": params[idx],
            "Best Avg Reward": avg_rewards[idx]
        })

    df2a = pd.DataFrame(table_2a)
    df2a.to_csv(os.path.join(out_dir, "table2a_best_parameter.csv"), index=False)

    table_2b = []
    for n in [5, 10, 20]:
        agents_n = {
            "ε-greedy (ε=0.1)": lambda n=n: EpsilonGreedyAgent(n, epsilon=0.1),
            "Optimistic (Q0=5, α=0.1)": lambda n=n: OptimisticGreedyAgent(n, q0=5.0, alpha=0.1),
            "UCB (c=1)": lambda n=n: UCBAgent(n, c=1.0),
            "Softmax (τ=0.1)": lambda n=n: SoftmaxAgent(n, tau=0.1),
            "Gradient bandit (α=0.1)": lambda n=n: GradientBanditAgent(n, alpha=0.1)
        }

        avg_reward_n, pct_opt_n = run_learning_curves(
            n_arms=n,
            n_steps=n_steps,
            n_runs=n_runs,
            agents=agents_n,
            seed=seed
        )

        for agent_name in avg_reward_n:
            table_2b.append({
                "n": n,
                "Agent": agent_name,
                "Final Avg Reward": avg_reward_n[agent_name][-1],
                "AUC Avg Reward": avg_reward_n[agent_name].mean(),
                "Final % Optimal": pct_opt_n[agent_name][-1],
                "AUC % Optimal": pct_opt_n[agent_name].mean()
            })

    df2b = pd.DataFrame(table_2b)
    df2b.to_csv(os.path.join(out_dir, "table2b_scaling_summary.csv"), index=False)

    print("\nProblem 2 tables saved:")
    print(" - table2a_best_parameter.csv")
    print(" - table2b_scaling_summary.csv")


In [8]:
# ============================================================
# Problem 3
# Action-Value Methods
# ============================================================

class ActionValueEpsilonGreedyAgent:
    def __init__(
        self,
        n_arms: int,
        epsilon: float = 0.1,
        update: str = 'sample_average',
        alpha: float = 0.1,
        W: Optional[int] = None
    ):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.update = update
        self.alpha = alpha
        self.W = W
        self.reset()

    def reset(self):
        self.Q = np.zeros(self.n_arms)
        self.N = np.zeros(self.n_arms, dtype=int)
        self.O = np.zeros(self.n_arms)
        self.reward_history = [[] for _ in range(self.n_arms)]

    def select_action(self, rng: np.random.Generator) -> int:
        if rng.random() < self.epsilon:
            return int(rng.integers(self.n_arms))
        return argmax_random_tie(self.Q, rng)

    def update_estimate(self, action: int, reward: float):
        self.N[action] += 1

        if self.update == 'sample_average':
            self.Q[action] += (reward - self.Q[action]) / self.N[action]

        elif self.update == 'constant_alpha':
            self.Q[action] += self.alpha * (reward - self.Q[action])

        elif self.update == 'unbiased_alpha':
            self.O[action] += self.alpha * (1 - self.O[action])
            beta = self.alpha / self.O[action]
            self.Q[action] += beta * (reward - self.Q[action])

        elif self.update == 'sliding_window':
            self.reward_history[action].append(reward)
            if len(self.reward_history[action]) > self.W:
                self.reward_history[action].pop(0)
            self.Q[action] = np.mean(self.reward_history[action])

        else:
            raise ValueError(f"Unknown update rule: {self.update}")


def run_bandit_experiment_stationary(
    agent_constructors: Dict[str, Callable[[], ActionValueEpsilonGreedyAgent]],
    n_arms: int = 10,
    n_steps: int = 2000,
    n_runs: int = 2000,
    seed: int = 42
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    rng_master = np.random.default_rng(seed)
    avg_rewards = {name: np.zeros(n_steps) for name in agent_constructors}
    pct_optimal = {name: np.zeros(n_steps) for name in agent_constructors}
    agent_names = list(agent_constructors.keys())

    for run in range(n_runs):
        q_star = rng_master.normal(0.0, 1.0, size=n_arms)
        optimal = int(np.argmax(q_star))

        envs = make_agent_envs(
            env_class=StationaryBandit,
            n_arms=n_arms,
            q_star=q_star,
            agent_names=agent_names
        )

        agents = {name: ctor() for name, ctor in agent_constructors.items()}
        rngs = {
            name: np.random.default_rng(seed + 100000 * (run + 1) + i)
            for i, name in enumerate(agent_names)
        }

        for t in range(n_steps):
            for name, agent in agents.items():
                rng = rngs[name]
                action = agent.select_action(rng)
                reward = envs[name].step(action, rng)
                agent.update_estimate(action, reward)

                avg_rewards[name][t] += reward
                pct_optimal[name][t] += int(action == optimal)

    for name in agent_constructors:
        avg_rewards[name] /= n_runs
        pct_optimal[name] = pct_optimal[name] / n_runs * 100.0

    return avg_rewards, pct_optimal


def run_bandit_experiment_nonstationary(
    agent_constructors: Dict[str, Callable[[], ActionValueEpsilonGreedyAgent]],
    n_arms: int = 10,
    n_steps: int = 2000,
    n_runs: int = 2000,
    seed: int = 42,
    random_walk_std: float = 0.01
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    """
    Shared drift path across estimators, independent reward noise per estimator.
    """
    rng_master = np.random.default_rng(seed)
    avg_rewards = {name: np.zeros(n_steps) for name in agent_constructors}
    pct_optimal = {name: np.zeros(n_steps) for name in agent_constructors}
    agent_names = list(agent_constructors.keys())

    for run in range(n_runs):
        q_star = rng_master.normal(0.0, 1.0, size=n_arms)

        envs = make_agent_envs(
            env_class=NonStationaryBandit,
            n_arms=n_arms,
            q_star=q_star,
            agent_names=agent_names,
            random_walk_std=random_walk_std
        )

        agents = {name: ctor() for name, ctor in agent_constructors.items()}
        rngs = {
            name: np.random.default_rng(seed + 100000 * (run + 1) + i)
            for i, name in enumerate(agent_names)
        }

        for t in range(n_steps):
            # All agents act on the same current q_star before drift is applied
            current_optimal = int(np.argmax(q_star))

            for name, agent in agents.items():
                rng = rngs[name]
                action = agent.select_action(rng)
                reward = envs[name].step(action, rng)
                agent.update_estimate(action, reward)

                avg_rewards[name][t] += reward
                pct_optimal[name][t] += int(action == current_optimal)

            # Shared drift applied after all agents observe rewards at step t
            drift = rng_master.normal(0.0, random_walk_std, size=n_arms)
            q_star = q_star + drift
            for name in agent_names:
                envs[name].apply_shared_drift(drift)

    for name in agent_constructors:
        avg_rewards[name] /= n_runs
        pct_optimal[name] = pct_optimal[name] / n_runs * 100.0

    return avg_rewards, pct_optimal


def plot_learning_curves_save(
    avg_rewards: Dict[str, np.ndarray],
    pct_optimal: Dict[str, np.ndarray],
    title_prefix: str,
    avg_filename: str,
    pct_filename: str,
    out_dir: str = "outputs"
) -> None:
    ensure_dir(out_dir)

    plt.figure(figsize=(10, 5))
    for name, vals in avg_rewards.items():
        plt.plot(smooth(vals), label=name, linewidth=2.5)

    plt.title(f'{title_prefix} — Average Reward')
    plt.xlabel('Steps')
    plt.ylabel('Average Reward')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, avg_filename), dpi=200)
    plt.close()

    plt.figure(figsize=(10, 5))
    for name, vals in pct_optimal.items():
        plt.plot(smooth(vals), label=name, linewidth=2.5)

    plt.title(f'{title_prefix} — % Optimal Action')
    plt.xlabel('Steps')
    plt.ylabel('% Optimal Action')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, pct_filename), dpi=200)
    plt.close()


def summary_table_problem3(avg_rewards, pct_optimal) -> pd.DataFrame:
    table = []
    for name in avg_rewards:
        table.append({
            'Estimator': name,
            'Final Avg Reward': avg_rewards[name][-1],
            'AUC Avg Reward': np.mean(avg_rewards[name]),
            'Final % Optimal': pct_optimal[name][-1],
            'AUC % Optimal': np.mean(pct_optimal[name])
        })
    return pd.DataFrame(table)


def run_problem3(out_dir: str, n_steps: int, n_runs: int, seed: int) -> None:
    n_arms = 10

    agent_constructors = {
        'Sample Average': lambda: ActionValueEpsilonGreedyAgent(
            n_arms, epsilon=0.1, update='sample_average'
        ),
        'Constant α': lambda: ActionValueEpsilonGreedyAgent(
            n_arms, epsilon=0.1, update='constant_alpha', alpha=0.1
        ),
        'Unbiased α': lambda: ActionValueEpsilonGreedyAgent(
            n_arms, epsilon=0.1, update='unbiased_alpha', alpha=0.1
        ),
        'Sliding Window': lambda: ActionValueEpsilonGreedyAgent(
            n_arms, epsilon=0.1, update='sliding_window', W=50
        ),
    }

    # Stationary
    avg_r_stat, pct_opt_stat = run_bandit_experiment_stationary(
        agent_constructors=agent_constructors,
        n_arms=n_arms,
        n_steps=n_steps,
        n_runs=n_runs,
        seed=seed
    )

    plot_learning_curves_save(
        avg_r_stat,
        pct_opt_stat,
        "Problem 3: Stationary Bandit",
        "fig4_q3_stationary_avg_reward.png",
        "fig4_q3_stationary_pct_optimal.png",
        out_dir=out_dir
    )

    # Non-stationary with shared drift
    avg_r_nonstat, pct_opt_nonstat = run_bandit_experiment_nonstationary(
        agent_constructors=agent_constructors,
        n_arms=n_arms,
        n_steps=n_steps,
        n_runs=n_runs,
        seed=seed,
        random_walk_std=0.01
    )

    plot_learning_curves_save(
        avg_r_nonstat,
        pct_opt_nonstat,
        "Problem 3: Nonstationary Bandit",
        "fig5_q3_nonstationary_avg_reward.png",
        "fig5_q3_nonstationary_pct_optimal.png",
        out_dir=out_dir
    )

    df_stat = summary_table_problem3(avg_r_stat, pct_opt_stat)
    df_nonstat = summary_table_problem3(avg_r_nonstat, pct_opt_nonstat)

    df_stat.to_csv(os.path.join(out_dir, "table3_stationary_summary.csv"), index=False)
    df_nonstat.to_csv(os.path.join(out_dir, "table3_nonstationary_summary.csv"), index=False)

    print("\nProblem 3 Stationary Environment Summary:")
    print(df_stat.to_string(index=False))
    print("\nProblem 3 Nonstationary Environment Summary:")
    print(df_nonstat.to_string(index=False))

In [9]:
# ============================================================
# Main
# ============================================================

def main():
    OUT_DIR = "outputs"
    N_STEPS = 2000
    N_RUNS = 2000
    SEED = 42

    ensure_dir(OUT_DIR)

    print("Running Problem 1...")
    run_problem1(OUT_DIR, N_STEPS, N_RUNS, SEED)

    print("\nRunning Problem 2...")
    run_problem2(OUT_DIR, N_STEPS, N_RUNS, SEED)

    print("\nRunning Problem 3...")
    run_problem3(OUT_DIR, N_STEPS, N_RUNS, SEED)

    print(f"\nAll figures and tables saved in '{OUT_DIR}/'")


if __name__ == "__main__":
    main()

Running Problem 1...

Problem 1 Summary Table: Sensitivity to Number of Arms
------------------------------------------------------
n  | Best Policy       | Runner-Up Policy  | Final Avg Reward | AUC Avg Reward | Std Avg Reward | Final % Optimal | AUC % Optimal | Std % Optimal
---+-------------------+-------------------+------------------+----------------+----------------+-----------------+---------------+--------------
5  | ε-greedy (ε=0.01) | ε-greedy (ε=0.1)  | 1.1101           | 1.048          | 0.0741         | 84.7            | 75.85         | 7.7          
10 | ε-greedy (ε=0.1)  | ε-greedy (ε=0.01) | 1.3908           | 1.3538         | 0.0967         | 83.6            | 77.1          | 9.78         
20 | ε-greedy (ε=0.1)  | ε-greedy (ε=0.01) | 1.6581           | 1.6024         | 0.1523         | 78.65           | 67.54         | 14.21        

Running Problem 2...

Problem 2 tables saved:
 - table2a_best_parameter.csv
 - table2b_scaling_summary.csv

Running Problem 3...

Problem